# Manuscript figure: cube to 6-tetrahedron decomposition

Body-diagonal decomposition of a voxel cube into six tetrahedra — the 3D analogue of the 2D TR–BL two-triangle split. Visual style mirrors `12a_3d-tetrahedral-check_concept.ipynb`, with generic $(x, y, z)$ axes and abstract vertex labels $A, B, \ldots, H$.

The body diagonal $\overline{AH}$ has Hamming distance $3$ in $(z, y, x)$ bits, so there are $3! = 6$ monotone edge paths from $A$ to $H$. Each path defines a tetrahedron whose four vertices are $A$, $H$, and the two intermediate cube corners traversed by that path. All six tetrahedra share the body diagonal as an edge.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

## Geometry

Eight cube corners $A, B, \ldots, H$ indexed by their $(z, y, x)$ offset bits (so $A$ is the origin and $H$ is the body-diagonal endpoint), and six tetrahedra $T_1, \ldots, T_6$ each formed by one of the $3!$ monotone edge paths from $A$ to $H$.

In [2]:
# 8 cube corners as (z, y, x) offsets, labeled A..H (vertex index i -> chr(65+i)).
CUBE_CORNERS = np.array([
    [0, 0, 0],  # A
    [0, 0, 1],  # B
    [0, 1, 0],  # C
    [0, 1, 1],  # D
    [1, 0, 0],  # E
    [1, 0, 1],  # F
    [1, 1, 0],  # G
    [1, 1, 1],  # H
], dtype=np.int8)

# 6 tetrahedra: each row gives the four corner indices along one monotone
# A -> ... -> H path (A and H are the body-diagonal endpoints; the middle
# two entries are the path's intermediate corners). Displayed as T_1..T_6
# in figures even though the underlying array is 0-indexed for Python.
TET_INDICES = np.array([
    [0, 1, 3, 7],   # T1
    [0, 1, 5, 7],   # T2
    [0, 2, 3, 7],   # T3
    [0, 2, 6, 7],   # T4
    [0, 4, 5, 7],   # T5
    [0, 4, 6, 7],   # T6
], dtype=np.int8)

# 12 edges of the unit cube
CUBE_EDGES = [
    (0, 1), (0, 2), (0, 4), (1, 3), (1, 5), (2, 3),
    (2, 6), (3, 7), (4, 5), (4, 6), (5, 7), (6, 7),
]

# Distinct, colour-blind-friendly hues for T_1..T_6 (0-indexed in code)
TET_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c',
              '#9467bd', '#17becf', '#bcbd22']

BODY_DIAGONAL_COLOR = '#d32f2f'


def xyz(corner_zyx):
    """(z, y, x) corner offset -> (x, y, z) for matplotlib 3D axes."""
    return float(corner_zyx[2]), float(corner_zyx[1]), float(corner_zyx[0])


def vertex_label(i):
    """Map vertex index 0..7 to its letter label A..H."""
    return chr(ord('A') + int(i))


def tet_label(t):
    """Map 0-indexed tet number 0..5 to its 1-indexed display label T_1..T_6."""
    return f'T_{{{t + 1}}}'

## Helpers

In [ ]:
def draw_cube_outline(ax, label_corners=True, edge_alpha=0.55,
                       body_diagonal=True, label_size=9,
                       highlight_vs=None, label_color='black'):
    """Draw the 12 cube edges, optionally label corners A..H and the body diagonal."""
    for i, j in CUBE_EDGES:
        x0, y0, z0 = xyz(CUBE_CORNERS[i])
        x1, y1, z1 = xyz(CUBE_CORNERS[j])
        ax.plot([x0, x1], [y0, y1], [z0, z1],
                color='#555555', lw=0.85, alpha=edge_alpha)
    if body_diagonal:
        x0, y0, z0 = xyz(CUBE_CORNERS[0])
        x1, y1, z1 = xyz(CUBE_CORNERS[7])
        ax.plot([x0, x1], [y0, y1], [z0, z1],
                color=BODY_DIAGONAL_COLOR, lw=1.8, alpha=0.85, zorder=8)
    if label_corners:
        highlight = set() if highlight_vs is None else set(highlight_vs)
        for i, c in enumerate(CUBE_CORNERS):
            x, y, z = xyz(c)
            is_h = i in highlight
            ax.scatter(x, y, z,
                       color='black' if is_h else '#555555',
                       s=28 if is_h else 18, zorder=11)
            ax.text(x + 0.04, y + 0.04, z + 0.06, f'${vertex_label(i)}$',
                    color=label_color, fontsize=label_size, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.16',
                              facecolor='#fff7e0' if is_h else 'white',
                              edgecolor='#444444' if is_h else '#bbbbbb',
                              linewidth=0.7, alpha=0.92),
                    ha='left', va='bottom', zorder=12)


def draw_tet(ax, t_idx, alpha=0.22, edge_lw=2.2, vertex_size=42,
             label_vertices=True, label_size=9):
    """Fill tetrahedron T_t, trace its monotone A -> ... -> H path, mark its 4 vertices."""
    inds = TET_INDICES[t_idx]
    color = TET_COLORS[t_idx]
    pts = np.array([xyz(CUBE_CORNERS[i]) for i in inds])

    # Translucent filled tet (4 faces)
    faces = [
        [pts[0], pts[1], pts[2]],
        [pts[0], pts[1], pts[3]],
        [pts[0], pts[2], pts[3]],
        [pts[1], pts[2], pts[3]],
    ]
    ax.add_collection3d(Poly3DCollection(
        faces, facecolor=color, alpha=alpha,
        edgecolor=color, linewidth=0.5))

    # Monotone path along three cube edges
    for k in range(3):
        x0, y0, z0 = pts[k]
        x1, y1, z1 = pts[k + 1]
        ax.plot([x0, x1], [y0, y1], [z0, z1],
                color=color, lw=edge_lw, alpha=0.95, zorder=9,
                solid_capstyle='round')

    # Vertices
    for vi in inds:
        x, y, z = xyz(CUBE_CORNERS[vi])
        ax.scatter(x, y, z, color=color, s=vertex_size, zorder=12,
                   edgecolor='black', linewidth=0.6)
        if label_vertices:
            ax.text(x + 0.04, y + 0.04, z + 0.06, f'${vertex_label(vi)}$',
                    color='black', fontsize=label_size, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.14',
                              facecolor='white', edgecolor=color,
                              linewidth=0.8, alpha=0.92),
                    ha='left', va='bottom', zorder=13)


def draw_axis_arrows(ax, anchor=(-0.05, 0.15), arrow_len=0.085,
                      with_labels=True, color='#222222', lw=1.8, label_size=11):
    """Draw a 2D compass widget at the lower-left of the 3D axes.

    Arrow directions are derived from the current 3D projection (so they
    correctly indicate which screen direction each +x/+y/+z axis points),
    but the compass itself is drawn entirely in **axes-fraction**
    coordinates — via ``ax.annotate(..., xycoords='axes fraction')`` and
    ``ax.text2D(..., transform=ax.transAxes)`` — so it can never be
    clipped by 3D rendering or by the figure boundary.

    The default anchor sits slightly LEFT of the axes (axes-fraction
    x = -0.05) so the compass clears the cube; the figure's left margin
    must be large enough to host it.
    """
    from mpl_toolkits.mplot3d import proj3d
    proj = ax.get_proj()
    px_o, py_o, _ = proj3d.proj_transform(0, 0, 0, proj)
    deltas = {}
    for label, end in (('x', (1.0, 0.0, 0.0)),
                        ('y', (0.0, 1.0, 0.0)),
                        ('z', (0.0, 0.0, 1.0))):
        px, py, _ = proj3d.proj_transform(end[0], end[1], end[2], proj)
        v = np.array([px - px_o, py - py_o])
        deltas[label] = v / np.linalg.norm(v)

    cx, cy = anchor
    label_offset = 0.025
    for label, d in deltas.items():
        tip_x = cx + d[0] * arrow_len
        tip_y = cy + d[1] * arrow_len
        ax.annotate('', xy=(tip_x, tip_y), xytext=(cx, cy),
                    xycoords='axes fraction',
                    arrowprops=dict(arrowstyle='-|>', color=color,
                                    lw=lw, shrinkA=0, shrinkB=0),
                    zorder=15, annotation_clip=False)
        if with_labels:
            ax.text2D(tip_x + d[0] * label_offset,
                      tip_y + d[1] * label_offset,
                      f'${label}$',
                      transform=ax.transAxes,
                      color=color, fontsize=label_size, fontweight='bold',
                      ha='center', va='center', zorder=16,
                      clip_on=False)


def style_3d_axes(ax, title='', zoom=1.55):
    # Expanded lower bound makes room for the compass widget on the first subplot
    # and keeps the cube the same on-screen size across all panels.
    ax.set_xlim(-0.3, 1.35)
    ax.set_ylim(-0.3, 1.35)
    ax.set_zlim(-0.3, 1.35)
    # Hide numerical tick labels and matplotlib's built-in axis labels.
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_zlabel('')
    # Remove the grey background panes + the gridlines drawn on them, and
    # paint matplotlib's per-axis spine lines transparent (they otherwise
    # extend past the cube into the negative range and read as a stray
    # intersecting line at the cube's back-bottom-right corner). We keep
    # the line objects alive so `bbox_inches='tight'` still works.
    transparent = (1.0, 1.0, 1.0, 0.0)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.fill = False
        axis.pane.set_edgecolor('white')
        axis.line.set_color(transparent)
    ax.grid(False)
    # zoom > 1 enlarges the rendered cube within the axes rectangle, reclaiming
    # the empty 3D-margin space that mpl reserves for tick labels by default.
    try:
        ax.set_box_aspect((1, 1, 1), zoom=zoom)
    except TypeError:
        # matplotlib < 3.6 fallback
        ax.set_box_aspect((1, 1, 1))
        ax.dist = 10.0 / zoom
    ax.view_init(elev=22, azim=-58)
    if title:
        # y < 0 places the subtitle below the axes; -0.08 is just under the
        # cube's visible bottom so the subtitle hugs the cube without overlap.
        # Larger fontsize keeps the LaTeX \frac fraction legible.
        ax.set_title(title, fontsize=13, y=-0.08)

## Figure 3: cube to 6-tetrahedron decomposition

2 × 4 grid of 3D panels:

- **(a)** Voxel cube with the 8 corners labeled $A, \ldots, H$ and the body diagonal $\overline{AH}$ in red.
- **(b)–(g)** Each tetrahedron $T_1, \ldots, T_6$ drawn individually inside the cube outline. Coloured edges trace the monotone path along cube edges from $A$ to $H$; the filled translucent volume is the tet; the red body diagonal is shown for context.
- **(h)** All six tetrahedra stacked at low opacity — they share the body diagonal, have disjoint interiors, and their union is the whole cube.

In [ ]:
def _raw_volume(t_idx):
    """Signed *raw* tetrahedron volume = scalar triple product / 6.

    No parity flip is applied — so even-permutation tets come out +1/6 and
    odd-permutation tets come out -1/6 on the reference cube.
    """
    inds = TET_INDICES[t_idx]
    pts = np.array([xyz(CUBE_CORNERS[i]) for i in inds])
    a, b, c, d = pts
    raw = float(np.dot(b - a, np.cross(c - a, d - a)))
    return raw / 6.0


def _tet_vertex_string(t_idx):
    """Render the tet's four monotone-path vertices as an *ordered simplex*
    in square-bracket notation, e.g. '[A, B, D, H]' — the standard
    convention for oriented simplices in simplicial topology / CG / FEM.
    """
    return '[' + ', '.join(vertex_label(int(i)) for i in TET_INDICES[t_idx]) + ']'


def _volume_symbol(v):
    """Render the signed volume as a LaTeX fraction +1/6 or -1/6."""
    return r'+\frac{1}{6}' if v > 0 else r'-\frac{1}{6}'


tet_volumes = [_raw_volume(t) for t in range(6)]

fig3 = plt.figure(figsize=(14.5, 6.4))

# ----- (a) cube with all 8 corners labeled + body diagonal -----
ax = fig3.add_subplot(2, 4, 1, projection='3d')
draw_cube_outline(ax, label_corners=True, body_diagonal=True,
                   highlight_vs=[0, 7])
style_3d_axes(ax, '(a) Voxel cube + body diagonal $\\overline{AH}$')
# Force a draw so ax.get_proj() returns the final projection before we
# build the 2D compass widget from it.
fig3.canvas.draw()
draw_axis_arrows(ax)

# ----- (b)-(g) individual tets T_1..T_6 with vertex ordering + signed volume -----
for t in range(6):
    ax = fig3.add_subplot(2, 4, 2 + t, projection='3d')
    draw_cube_outline(ax, label_corners=False, body_diagonal=True)
    draw_tet(ax, t)
    title = (f'({chr(98 + t)}) ${tet_label(t)} = {_tet_vertex_string(t)}$,  '
             f'$\\mathrm{{vol}}({tet_label(t)}) = {_volume_symbol(tet_volumes[t])}$')
    style_3d_axes(ax, title)

# ----- (h) all six tets stacked -----
ax = fig3.add_subplot(2, 4, 8, projection='3d')
draw_cube_outline(ax, label_corners=False, body_diagonal=True)
for t in range(6):
    inds = TET_INDICES[t]
    pts = np.array([xyz(CUBE_CORNERS[i]) for i in inds])
    faces = [
        [pts[0], pts[1], pts[2]],
        [pts[0], pts[1], pts[3]],
        [pts[0], pts[2], pts[3]],
        [pts[1], pts[2], pts[3]],
    ]
    ax.add_collection3d(Poly3DCollection(
        faces, facecolor=TET_COLORS[t], alpha=0.13,
        edgecolor=TET_COLORS[t], linewidth=0.5))
style_3d_axes(ax, '(h) All six tetrahedra stacked')

# left=0.03 reserves space for the panel-(a) compass widget which is
# anchored slightly left of the axes (axes-fraction x=-0.05). The pixel
# bbox crop trims any leftover margin at save time.
fig3.subplots_adjust(left=0.03, right=1.0, top=1.0, bottom=0.08,
                     wspace=0.0, hspace=0.20)

plt.show()

## Save

Writes PNG and PDF versions of the figure to `./output/` (relative to this notebook).

In [5]:
import matplotlib.transforms as mtransforms


def _rendered_content_bbox(fig, pad_inches=0.05, near_white=250):
    """Tight bbox (in figure-inches) of the *visibly drawn pixels*.

    ``bbox_inches='tight'`` includes the invisible 3D-decoration band
    that mpl reserves around each Axes3D, so it leaves whitespace at the
    bottom. Here we rasterize the figure to its own canvas, find the
    bounding box of non-white pixels, and convert that back to figure
    inches. The result is what gets handed to ``savefig`` so the saved
    PDF / PNG are cropped to the actual content.
    """
    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba())
    rgb = buf[..., :3]
    mask = (rgb < near_white).any(axis=-1)
    if not mask.any():
        return None
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    h_px, w_px = buf.shape[:2]
    fw, fh = fig.get_size_inches()
    # matplotlib's figure y-origin is at the bottom; the canvas buffer's
    # row 0 is at the top. Convert pixel rows -> figure inches.
    top_in    = fh * (1 - rows[0]    / h_px)
    bottom_in = fh * (1 - rows[-1]   / h_px)
    left_in   = fw * (cols[0]        / w_px)
    right_in  = fw * (cols[-1]       / w_px)
    return mtransforms.Bbox.from_extents(
        max(0.0, left_in   - pad_inches),
        max(0.0, bottom_in - pad_inches),
        min(fw,  right_in  + pad_inches),
        min(fh,  top_in    + pad_inches),
    )


output_dir = 'output/tet_decomposition'
os.makedirs(output_dir, exist_ok=True)

bbox3 = _rendered_content_bbox(fig3)

fig3.savefig(os.path.join(output_dir, 'tet_decomposition.pdf'),
             bbox_inches=bbox3)
fig3.savefig(os.path.join(output_dir, 'tet_decomposition.png'),
             dpi=300, bbox_inches=bbox3)

print(f'Saved figure to {os.path.abspath(output_dir)}')

Saved figure to c:\Users\Andy\Documents\GitHub\UCI-iGravi\deformation-field-processing\notebooks\manuscript\output\tet_decomposition
